# BBDE 301366 — DQ Check

Data availability checks for BBDE 301366.
- Union temp views as source (not individual LOB tables)
- Same `check_column()` / `insertDQTable()` pattern as TDI
- Table/column names verified against original BBDE notebooks

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/GAMLConnections

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/Settings

---
## Shared Functions

In [ ]:
from datetime import date

TABLE_NAME = SNAPSHOT_CATALOGUE_NAME + '.' + TABLE_NAME_DATA_AVA_SEG
today_date = str(date.today())

lob_id = '301366'
lob_desc = 'BBDE'

def insertDQTable(SNAPSHOT_CATALOGUE_NAME, TABLE_NAME_DATA_AVA_SEG,
                  lob_id, cde_no, cde_desc, source, src_table_name,
                  data_element, availability_pct, today_date):
    query = ("INSERT INTO " + SNAPSHOT_CATALOGUE_NAME + "." + TABLE_NAME_DATA_AVA_SEG
             + " VALUES('" + lob_id + "','" + cde_no + "','" + cde_desc + "','"
             + source + "','" + src_table_name + "','" + data_element + "','"
             + str(availability_pct) + "','" + today_date + "')")
    spark.sql(query)
    return True

def check_column(cde_no, cde_desc, source, src_table, column):
    try:
        query = f"""
            SELECT count(1) as total,
                   count(CASE WHEN cast({column} as STRING) IS NOT NULL
                         AND trim(cast({column} as STRING)) != '' THEN 1 END) as valid
            FROM {src_table}
        """
        result = spark.sql(query).collect()[0]
        total, valid = result[0], result[1]
        pct = round(100.0 * valid / total, 2) if total > 0 else 0
        insertDQTable(SNAPSHOT_CATALOGUE_NAME, TABLE_NAME_DATA_AVA_SEG,
                      lob_id, cde_no, cde_desc, source, src_table,
                      column, pct, today_date)
        print(f"  {column}: {valid}/{total} = {pct}%")
    except Exception as e:
        print(f"  {column}: ERROR - {str(e)}")

print(f'DQ Check for {lob_id} ({lob_desc})')
print(f'Results table: {TABLE_NAME}')
print(f'Run date: {today_date}')

---
## Step 0: Verify Table & Column Names

In [ ]:
# ============================================================
# Verify tables exist and show ALL columns
# ============================================================
tables_to_check = [
    'RA_FY_2025.EDB_FULL_GEN_BUSINESS',
    'RA_FY_2025.PSI_FULL_GEN_BUSINESS',
    'RA_FY_2025.PL_FULL_GEN',
    'ra_fy_2025.resl_full_gen_5',
    'RA_FY_2025.CC_FULL_GEN',
    'RA_FY_2025.TDIS_FULL_GEN_BUSINESS',
]

for t in tables_to_check:
    print(f'\n{"=" * 70}')
    print(f'{t}')
    print(f'{"=" * 70}')
    try:
        cols = spark.sql(f'DESCRIBE TABLE {t}').collect()
        for row in cols:
            print(f'  {row[0]:<40} {row[1]}')
    except Exception as e:
        print(f'  ERROR: {e}')

# Centralized tables
print(f'\n{"=" * 70}')
print('Centralized tables')
print(f'{"=" * 70}')
for t in ['rafy2025_centralized.Customer_scorable_rated_CA',
          'rafy2025_centralized.CRR_Scorable_Cust_CA',
          'rafy2025_centralized.cde_sd_6_1yr_fy2025',
          'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details',
          'rafy2025_centralized.td_sar_cde_3_18_2025',
          'rafy2025_centralized.cde_3_19_lctr']:
    try:
        cnt = spark.sql(f'SELECT count(1) FROM {t}').collect()[0][0]
        print(f'  {t:<60} {cnt:>12,} rows')
    except Exception as e:
        print(f'  {t:<60} ERROR: {e}')

---
## Step 1: Create Union Temp Views
Table/column names from original BBDE_Segment_301366_QUERIES_FY25 notebook.

In [ ]:
# ============================================================
# Retail Union: EDB + PSI + PL + RESL + CC + TDIS
# Column names verified from DESCRIBE TABLE + original queries
# ============================================================
spark.sql("""
    CREATE OR REPLACE TEMP VIEW bbde_retail_union AS
    -- EDB (301270) ~11,186,444 customers
    SELECT cust_id,
           upper(concat(first_na, ' ', last_na)) AS fulln,
           birth_dt, acct_id, 'EDB' AS lob_source
    FROM RA_FY_2025.EDB_FULL_GEN_BUSINESS
    WHERE cbs_efectv_dt BETWEEN '2024-11-01' AND '2025-10-31'
      AND cas_efectv_dt BETWEEN '2024-11-01' AND '2025-10-31'
      AND acct_type_id <> '1211'
      AND cust_type_cd = 2235
      AND lifecy_cd = 115
    UNION ALL
    -- PSI (301275) ~3,449,972 customers
    SELECT cust_cust_id AS cust_id,
           upper(concat(first_na, ' ', last_na)) AS fulln,
           birth_dt, ca_acct_id AS acct_id, 'PSI' AS lob_source
    FROM RA_FY_2025.PSI_FULL_GEN_BUSINESS
    WHERE ahb_lifecy_cd != 115
      AND ahb_efectv_dt BETWEEN '2024-11-01' AND '2025-10-31'
    UNION ALL
    -- PL (301273) ~3,265,363 customers
    SELECT cas_cust_id AS cust_id,
           upper(concat(first_na, ' ', last_na)) AS fulln,
           birth_dt, cas_acct_id AS acct_id, 'PL' AS lob_source
    FROM RA_FY_2025.PL_FULL_GEN
    WHERE cbs_efectv_dt BETWEEN '2024-11-01' AND '2025-10-31'
    UNION ALL
    -- RESL (301274) ~2,360,416 customers
    SELECT cas_cust_id AS cust_id,
           upper(concat(first_na, ' ', last_na)) AS fulln,
           birth_dt, cas_acct_id AS acct_id, 'RESL' AS lob_source
    FROM ra_fy_2025.resl_full_gen_5
    WHERE cbs_efectv_dt BETWEEN '2024-11-01' AND '2025-10-31'
      AND lifecy_cd = 115
    UNION ALL
    -- CC (301064) ~7,152,843 customers
    SELECT ca_cust_id AS cust_id,
           upper(concat(first_na, ' ', last_na)) AS fulln,
           birth_dt, ca_acct_id AS acct_id, 'CC' AS lob_source
    FROM RA_FY_2025.CC_FULL_GEN
    WHERE ahb_efectv_dt BETWEEN '2024-11-01' AND '2025-10-31'
      AND (ahb_appl_stat_cd IN (0,23,1218,1220,10943,10944,18945)
           OR (ahb_appl_stat_cd IN (14,1223,1227,1257,1420)
               AND (ahb_moend_pria_am + ahb_moend_accr_int_am) > 0))
    UNION ALL
    -- TDIS (301087) ~1,708,626 customers
    SELECT ca_cust_id AS cust_id,
           upper(concat(first_na, ' ', last_na)) AS fulln,
           birth_dt, ca_acct_id AS acct_id, 'TDIS' AS lob_source
    FROM RA_FY_2025.TDIS_FULL_GEN_BUSINESS
    WHERE ahb_efectv_dt BETWEEN '2024-11-01' AND '2025-10-31'
      AND ahb_lifecy_cd IN (114, 116, 117)
""")
print(f'bbde_retail_union: {spark.sql("SELECT count(1) FROM bbde_retail_union").collect()[0][0]:,} rows')

In [ ]:
# ============================================================
# Business Union: SBB_DP + COM_DP + SBB_Credit + COM_Credit
# Column: cif_business_customer_key_token (from original notebook)
# ============================================================
spark.sql("""
    CREATE OR REPLACE TEMP VIEW bbde_business_union AS
    -- SBB deposit (1,009,789)
    SELECT cif_business_customer_key_token AS cust_id,
           'SBB_DP' AS lob_source
    FROM ra_fy_2025.SBB_DP_Final
    UNION ALL
    -- SBB Credit (263,890)
    SELECT cif_business_customer_key_token AS cust_id,
           'SBB_CR' AS lob_source
    FROM ra_fy_2025.sbb_credit_2025
    UNION ALL
    -- COM deposit (95,979)
    SELECT cif_business_customer_key_token AS cust_id,
           'COM_DP' AS lob_source
    FROM ra_fy_2025.COM_DP_Final
    UNION ALL
    -- COM credit (49,911)
    SELECT cif_business_customer_key_token AS cust_id,
           'COM_CR' AS lob_source
    FROM ra_fy_2025.com_credit_2025
""")
print(f'bbde_business_union: {spark.sql("SELECT count(1) FROM bbde_business_union").collect()[0][0]:,} rows')

In [ ]:
# ============================================================
# TDS Union: TDS_201037 (GM) + TDS_201042 (TF)
# ============================================================
spark.sql("""
    CREATE OR REPLACE TEMP VIEW bbde_tds_union AS
    -- TDS Global Markets (201037) ~44,322
    SELECT DISTINCT ROOT_GT_ID AS cust_id,
           'TDS_GM' AS lob_source
    FROM RA_FY25_VIEW.TDS_201037
    WHERE ROOT_CLOSE_DATE IS NULL
       OR ROOT_CLOSE_DATE = 'NULL'
       OR date(ROOT_CLOSE_DATE) <= date('2025-10-31')
    UNION ALL
    -- TDS Trade Finance (201042) ~4,991
    SELECT DISTINCT ROOT_GT_ID AS cust_id,
           'TDS_TF' AS lob_source
    FROM RA_FY25_VIEW.TDS_201042
    WHERE ROOT_CLOSE_DATE IS NULL
       OR ROOT_CLOSE_DATE = 'NULL'
       OR date(ROOT_CLOSE_DATE) <= date('2025-10-31')
""")
print(f'bbde_tds_union: {spark.sql("SELECT count(1) FROM bbde_tds_union").collect()[0][0]:,} rows')

In [ ]:
# ============================================================
# TDW: PB_VIEW + tdw_all_account (DI)
# ============================================================
spark.sql("""
    CREATE OR REPLACE TEMP VIEW bbde_tdw_union AS
    -- TDW PB (~134,639)
    SELECT DISTINCT cust_id,
           'TDW_PB' AS lob_source
    FROM RA_FY25_VIEW.PB_VIEW
    UNION ALL
    -- TDW DI (~13,073,036 rows, ~2,012,570 distinct mstr_rec_id)
    SELECT DISTINCT mstr_rec_id AS cust_id,
           'TDW_DI' AS lob_source
    FROM ra_fy25_view.tdw_all_account
    WHERE line_of_business = 'DI'
      AND (account_close_date = '' OR account_close_date > '2024-10-31')
      AND pty_typ_id IN (1006000, 1006020)
""")
print(f'bbde_tdw_union: {spark.sql("SELECT count(1) FROM bbde_tdw_union").collect()[0][0]:,} rows')

In [ ]:
# ============================================================
# Dedup / Common Client tables
# ============================================================
spark.sql("""
    CREATE OR REPLACE TEMP VIEW bbde_dedup_union AS
    -- BBDE retail common (~1,575,668)
    SELECT cust_id, 'BBDE_COMMON' AS source
    FROM ra_adido_2025.bbde_retail1_common_client_final
    UNION ALL
    -- TDS retail common
    SELECT REPLACE(Customer_GoIdifier_ID, '.', '') AS cust_id,
           'TDS_COMMON' AS source
    FROM ra_adido_2025.tds_retail_common_clients
""")
print(f'bbde_dedup_union: {spark.sql("SELECT count(1) FROM bbde_dedup_union").collect()[0][0]:,} rows')

---
## Step 2: Segment DQ Checks

In [ ]:
# ============================================================
# Retail Union checks
# ============================================================
print('bbde_retail_union')
print('=' * 60)
check_column('SD1,1.2,1.5', 'Segment', 'Union:Retail', 'bbde_retail_union', 'cust_id')
check_column('SD1,1.7,3.17,3.18', 'Segment', 'Union:Retail', 'bbde_retail_union', 'fulln')
check_column('3.17,6.5A,6.5B', 'Segment', 'Union:Retail', 'bbde_retail_union', 'birth_dt')
check_column('4.1A', 'Segment', 'Union:Retail', 'bbde_retail_union', 'acct_id')
check_column('Debug', 'Segment', 'Union:Retail', 'bbde_retail_union', 'lob_source')

In [ ]:
# ============================================================
# Business Union checks
# ============================================================
print('bbde_business_union')
print('=' * 60)
check_column('SD1,1.2,1.5', 'Segment', 'Union:Business', 'bbde_business_union', 'cust_id')

In [ ]:
# ============================================================
# TDS Union checks
# ============================================================
print('bbde_tds_union')
print('=' * 60)
check_column('SD1,1.2,1.3,1.4', 'Segment', 'Union:TDS', 'bbde_tds_union', 'cust_id')

In [ ]:
# ============================================================
# TDW Union checks
# ============================================================
print('bbde_tdw_union')
print('=' * 60)
check_column('SD1', 'Segment', 'Union:TDW', 'bbde_tdw_union', 'cust_id')

In [ ]:
# ============================================================
# Dedup Union checks
# ============================================================
print('bbde_dedup_union')
print('=' * 60)
check_column('Dedup', 'Segment', 'Union:Dedup', 'bbde_dedup_union', 'cust_id')

---
## Step 3: Reference / Lookup Tables

In [ ]:
# ============================================================
# EDW Customer (for dedup convert)
# ============================================================
print('ra_fy_2025.edw_customer')
print('=' * 60)
check_column('Dedup', 'Reference', 'Rahona', 'ra_fy_2025.edw_customer', 'edw_customer_id')
check_column('Dedup', 'Reference', 'Rahona', 'ra_fy_2025.edw_customer', 'bor_account_id')

---
## Step 4: Centralized Metric Sources
Table names verified from original BBDE 301366 Centralized notebook.

In [ ]:
# ============================================================
# CRR: Customer_scorable_rated_CA (CDE 1.2, 1.3, 1.4)
# ============================================================
print('rafy2025_centralized.Customer_scorable_rated_CA')
print('=' * 60)
check_column('1.2,1.3,1.4,1.5', 'Centralized', 'Rahona',
             'rafy2025_centralized.Customer_scorable_rated_CA', 'v_entity_id')
check_column('1.2,1.3,1.4', 'Centralized', 'Rahona',
             'rafy2025_centralized.Customer_scorable_rated_CA', 'risk_rating')
check_column('1.2,1.3,1.4,1.5', 'Centralized', 'Rahona',
             'rafy2025_centralized.Customer_scorable_rated_CA', 'v_cust_type_cd')

In [ ]:
# ============================================================
# CRR Scorable: CRR_Scorable_Cust_CA (CDE 1.1, TDW sections)
# ============================================================
print('rafy2025_centralized.CRR_Scorable_Cust_CA')
print('=' * 60)
check_column('1.1,TDW', 'Centralized', 'Rahona',
             'rafy2025_centralized.CRR_Scorable_Cust_CA', 'v_entity_id')

In [ ]:
# ============================================================
# SD2 (PEP): pep_list_2025_exploded
# ============================================================
print('ra_adido_2025.pep_list_2025_exploded')
print('=' * 60)
check_column('SD2', 'Centralized', 'ADIDO', 'ra_adido_2025.pep_list_2025_exploded', 'entity')
check_column('SD2', 'Centralized', 'ADIDO', 'ra_adido_2025.pep_list_2025_exploded', 'first_nm')
check_column('SD2', 'Centralized', 'ADIDO', 'ra_adido_2025.pep_list_2025_exploded', 'last_nm')

In [ ]:
# ============================================================
# CDE 1.7: true_sanction_match_2025
# ============================================================
print('ra_adido_2025.true_sanction_match_2025')
print('=' * 60)
check_column('1.7', 'Centralized', 'ADIDO', 'ra_adido_2025.true_sanction_match_2025', 'Customer')
check_column('1.7', 'Centralized', 'ADIDO', 'ra_adido_2025.true_sanction_match_2025', 'CustomerType')

In [ ]:
# ============================================================
# CDE 1.8: customer_sanctioned_blocked_property_2025
# ============================================================
print('ra_adido_2025.customer_sanctioned_blocked_property_2025')
print('=' * 60)
check_column('1.8', 'Centralized', 'ADIDO',
             'ra_adido_2025.customer_sanctioned_blocked_property_2025', 'CUSTOMERIMPACTED')
check_column('1.8', 'Centralized', 'ADIDO',
             'ra_adido_2025.customer_sanctioned_blocked_property_2025', 'ACCOUNTBLOCKED')

In [ ]:
# ============================================================
# SD6 (LYR): cde_sd_6_1yr_fy2025  (note: 1yr not lyr)
# ============================================================
print('rafy2025_centralized.cde_sd_6_1yr_fy2025')
print('=' * 60)
check_column('SD6', 'Centralized', 'Rahona',
             'rafy2025_centralized.cde_sd_6_1yr_fy2025', 'v_entity_id')

In [ ]:
# ============================================================
# CDE 3.17 (UTR): TD_UTR_CDE_3_17_2025_Cust_details (345,128 rows)
# ============================================================
print('rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details')
print('=' * 60)
check_column('3.17', 'Centralized', 'Rahona',
             'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'cust_no')
check_column('3.17', 'Centralized', 'Rahona',
             'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'full_name')
check_column('3.17', 'Centralized', 'Rahona',
             'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'cust_type_mn')
check_column('3.17', 'Centralized', 'Rahona',
             'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'birth_dt')
check_column('3.17', 'Centralized', 'Rahona',
             'rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'account_type')

In [ ]:
# ============================================================
# CDE 3.18 (STR/SAR): td_sar_cde_3_18_2025 (8,225 rows)
# ============================================================
print('rafy2025_centralized.td_sar_cde_3_18_2025')
print('=' * 60)
check_column('3.18', 'Centralized', 'Rahona',
             'rafy2025_centralized.td_sar_cde_3_18_2025', 'STR_Admin_Matched_CustomerID')
check_column('3.18', 'Centralized', 'Rahona',
             'rafy2025_centralized.td_sar_cde_3_18_2025', 'STR_Admin_Matched_Customer_Type')
check_column('3.18', 'Centralized', 'Rahona',
             'rafy2025_centralized.td_sar_cde_3_18_2025', 'STR_LETUA_Customer_Account_Number')
check_column('3.18', 'Centralized', 'Rahona',
             'rafy2025_centralized.td_sar_cde_3_18_2025', 'customer_first_name')
check_column('3.18', 'Centralized', 'Rahona',
             'rafy2025_centralized.td_sar_cde_3_18_2025', 'customer_last_name')

In [ ]:
# ============================================================
# CDE 3.19 (LCTR): cde_3_19_lctr (43,744,859 rows)
# ============================================================
print('rafy2025_centralized.cde_3_19_lctr')
print('=' * 60)
check_column('3.19', 'Centralized', 'Rahona',
             'rafy2025_centralized.cde_3_19_lctr', 'account_number')
check_column('3.19', 'Centralized', 'Rahona',
             'rafy2025_centralized.cde_3_19_lctr', 'client_name')

---
## Step 5: Row Count Summary

In [ ]:
tables = [
    ('RA_FY_2025.EDB_FULL_GEN_BUSINESS', 'Retail: EDB'),
    ('RA_FY_2025.PSI_FULL_GEN_BUSINESS', 'Retail: PSI'),
    ('RA_FY_2025.PL_FULL_GEN', 'Retail: PL'),
    ('ra_fy_2025.resl_full_gen_5', 'Retail: RESL'),
    ('RA_FY_2025.CC_FULL_GEN', 'Retail: CC'),
    ('RA_FY_2025.TDIS_FULL_GEN_BUSINESS', 'Retail: TDIS'),
    ('ra_fy_2025.SBB_DP_Final', 'Business: SBB DP'),
    ('ra_fy_2025.COM_DP_Final', 'Business: COM DP'),
    ('ra_fy_2025.sbb_credit_2025', 'Business: SBB Credit'),
    ('ra_fy_2025.com_credit_2025', 'Business: COM Credit'),
    ('RA_FY25_VIEW.TDS_201037', 'TDS: Global Markets'),
    ('RA_FY25_VIEW.TDS_201042', 'TDS: Trade Finance'),
    ('RA_FY25_VIEW.PB_VIEW', 'TDW: PB'),
    ('ra_fy25_view.tdw_all_account', 'TDW: DI (all)'),
    ('ra_adido_2025.bbde_retail1_common_client_final', 'Dedup: BBDE Common'),
    ('ra_adido_2025.tds_retail_common_clients', 'Dedup: TDS Common'),
    ('ra_fy_2025.edw_customer', 'Dedup: EDW'),
    ('rafy2025_centralized.Customer_scorable_rated_CA', 'CRR: Rated'),
    ('rafy2025_centralized.CRR_Scorable_Cust_CA', 'CRR: Scorable'),
    ('ra_adido_2025.pep_list_2025_exploded', 'SD2: PEP'),
    ('ra_adido_2025.true_sanction_match_2025', 'CDE 1.7: Sanctions'),
    ('ra_adido_2025.customer_sanctioned_blocked_property_2025', 'CDE 1.8: Blocked'),
    ('rafy2025_centralized.cde_sd_6_1yr_fy2025', 'SD6: LYR'),
    ('rafy2025_centralized.TD_UTR_CDE_3_17_2025_Cust_details', 'CDE 3.17: UTR'),
    ('rafy2025_centralized.td_sar_cde_3_18_2025', 'CDE 3.18: STR/SAR'),
    ('rafy2025_centralized.cde_3_19_lctr', 'CDE 3.19: LCTR'),
    ('bbde_retail_union', 'UNION: Retail'),
    ('bbde_business_union', 'UNION: Business'),
    ('bbde_tds_union', 'UNION: TDS'),
    ('bbde_tdw_union', 'UNION: TDW'),
    ('bbde_dedup_union', 'UNION: Dedup'),
]

print(f'{"Table":<65} {"Description":<25} {"Row Count":>15}')
print('=' * 110)
for table, desc in tables:
    try:
        cnt = spark.sql(f'SELECT count(1) FROM {table}').collect()[0][0]
        print(f'{table:<65} {desc:<25} {cnt:>15,}')
    except Exception as e:
        print(f'{table:<65} {desc:<25} {"ERROR":>15}')

---
## Step 6: Summary

In [ ]:
results = spark.sql(f"""
    SELECT * FROM {TABLE_NAME}
    WHERE lob_id = '{lob_id}'
      AND run_date = '{today_date}'
    ORDER BY src_table_name, data_element
""")
print(f'Total DQ checks: {results.count()}')
display(results)

In [ ]:
for view in ['bbde_retail_union', 'bbde_business_union', 'bbde_tds_union',
             'bbde_tdw_union', 'bbde_dedup_union']:
    spark.sql(f'DROP VIEW IF EXISTS {view}')
    print(f'Dropped: {view}')
print('Cleanup complete.')